# HW2 Part 1 — Data cleaning and validation of `ohlc_hw`

**Input:** `data/data.db` table `ohlc_hw` (raw, never modified). **Output:** table `ohlc_hw_cleaned`
(plus audit tables `ohlc_hw_quarantine`, `ohlc_hw_quality_report`). Reference for dates and
adjustment factors: supplied table `ohlc`.

| Stage | Rows |
|---|---|
| Raw `ohlc_hw` | 397,711 |
| Exact duplicates removed (after ticker normalization) | 29,460 |
| Quarantined (`open = 0` on first day; no earlier price, reference also 0) | 5 |
| Retained in `ohlc_hw_cleaned` | 368,246 |

**Key findings and decisions**
- Tickers: 150 HTML-wrapped labels → 30 tickers (tags stripped, trimmed, upper-cased) *before* grouping or dedup. Original label kept in `raw_ticker`.
- No conflicting `(ticker, ts)` keys; no missing sessions vs each ticker's `ohlc` dates (detector demonstrated on an injected gap).
- Raw OHLC/volume equal the reference on every row; the damage was in `adj_close`: 18,374 missing and 10,417 inconsistent with the reference factor → rebuilt as `factor × close` using the date-specific reference factor. One-day reference factor blips (e.g. AAPL 1990-10-19) use the neighbouring factor.
- 3 one-day whole-row spikes (TRV, price ×2, zero volume, fully reversed next day) are confirmed errors → OHLC interpolated from neighbours.
- All other 3σ return flags are kept and flagged (`extreme_return_flag`), not deleted. Repairing the spikes reduced σ, so the post-repair flag count rises (5,056 → 5,176).
- Impossible OHLC relationships: Option 1 (NaN the faulty field, forward-fill within ticker), then widen high/low to contain open/close; close is never moved.
- Severity: **high** = value cannot be true; **medium** = may be true but needs review / blocks metrics; **low** = plausible, price unaffected.
- Known limitation: the reference factor for DIS in Jan–Feb 1974 alternates between ~0.621 and ~0.663, so those adjusted prices are approximate (±7%).

## 1. Load the raw data
Read `ohlc_hw` from `data/data.db`. The raw table is never modified; all work happens on copies.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

# data.db is distributed via bCourse and gitignored; it sits next to this notebook
DB_PATH = Path("data/data.db")
print("DB exists:", DB_PATH.exists())

# Read the raw table once. It is only ever read, never written back.
with sqlite3.connect(DB_PATH) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

print(raw.shape)
raw.head()

DB exists: True
(397711, 8)


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


## 2. First look: columns, types, ranges
What each column means, plus types, missing counts, the date range and the raw ticker labels. The ticker labels turn out to be wrapped in HTML.

In [2]:
COLUMN_MEANINGS = {
    "ts": "observation date (text, YYYY-MM-DD)",
    "ticker": "stock symbol as delivered by the vendor (may contain markup/whitespace)",
    "open": "raw opening price",
    "high": "raw daily high",
    "low": "raw daily low",
    "close": "raw closing price",
    "adj_close": "vendor split/dividend-adjusted close",
    "volume": "shares traded",
}
# Data dictionary: what each column is supposed to mean
display(pd.Series(COLUMN_MEANINGS, name="meaning").to_frame())

# Types, non-null counts and summary statistics (spot missing values and odd ranges)
raw.info()
display(raw.describe())

# Check the date column: overall range and any strings that are not valid dates
ts_parsed = pd.to_datetime(raw["ts"], errors="coerce")
print("Date range:", ts_parsed.min().date(), "to", ts_parsed.max().date())
print("Unparseable dates:", ts_parsed.isna().sum())

# Tickers: far more distinct labels than real symbols means the labels are dirty
raw_tickers = raw["ticker"].value_counts()
print("Distinct raw ticker strings:", raw_tickers.size)
print(sorted(raw_tickers.index)[:40])

,meaning
ts,"observation date (text, YYYY-MM-DD)"
ticker,stock symbol as delivered by the vendor (may c...
open,raw opening price
high,raw daily high
low,raw daily low
close,raw closing price
adj_close,vendor split/dividend-adjusted close
volume,shares traded


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Date range: 1962-01-02 to 2024-12-06
Unparseable dates: 0
Distinct raw ticker strings: 150
['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span></a>', '<a target="_blank"><span

## 3. Normalize tickers and remove duplicates
Tickers are cleaned **before** grouping or de-duplicating; otherwise the same stock would be split across several labels. The original label is kept in `raw_ticker`.

- **Exact duplicates** (every value identical) are dropped.
- **Conflicting keys** (same `ticker, ts` with different values) would need a rule to choose; there are none.

In [3]:
def normalize_ticker(labels: pd.Series) -> pd.Series:
    """Strip HTML tags and whitespace, then upper-case vendor ticker labels."""
    return (labels.astype("string")
                  .str.replace(r"<[^>]+>", "", regex=True)
                  .str.strip()
                  .str.upper())

work = raw.copy()
work["raw_ticker"] = work["ticker"]                     # keep the original label
work["ticker"] = normalize_ticker(work["ticker"])
work["ts"] = pd.to_datetime(work["ts"]).dt.strftime("%Y-%m-%d")   # ISO text

# Audit table: which raw labels were merged into each clean ticker
ticker_audit = (work.groupby(["ticker", "raw_ticker"]).size()
                    .rename("rows").reset_index())
print("Raw labels:", work["raw_ticker"].nunique(), "→ tickers:", work["ticker"].nunique())
print(sorted(work["ticker"].unique()))

# Exact duplicates: identical in every value column -> safe to drop (keep the first)
value_cols = ["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume"]
exact_dup = work.duplicated(subset=value_cols, keep="first")
print("Exact duplicate rows:", exact_dup.sum())
dedup = work.loc[~exact_dup].copy()

# Conflicts: same (ticker, ts) but different values -> would need a rule to choose one
key = ["ticker", "ts"]
conflicts = dedup.loc[dedup.duplicated(key, keep=False)].sort_values(key)
print("Rows in conflicting (ticker, ts) keys:", len(conflicts))
display(conflicts.head(10))
display(ticker_audit.groupby("ticker")["raw_ticker"].nunique().rename("raw_label_variants").to_frame().T)

Raw labels: 150 → tickers: 30
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']
Exact duplicate rows: 29460
Rows in conflicting (ticker, ts) keys: 0


,ts,open,high,low,close,adj_close,volume,ticker,raw_ticker


ticker,AAPL,AMGN,AMZN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,...,MSFT,NKE,NVDA,PG,SHW,TRV,UNH,V,VZ,WMT
raw_label_variants,5,5,5,5,5,5,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5


## 4. Missing values and missing trading sessions
A missing session is a date where the stock traded (it appears in the reference `ohlc` table) but `ohlc_hw` has no row. The detector is tested on a date that was removed on purpose.

In [4]:
def missing_cells(frame: pd.DataFrame) -> pd.DataFrame:
    """Count missing values per column."""
    counts = frame.isna().sum()
    return counts.rename("missing").to_frame().assign(pct=lambda d: d["missing"] / len(frame))

def missing_sessions(data: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    """Return (ticker, ts) pairs present in the reference calendar but absent from data.

    The reference calendar for each ticker is its own dates in the supplied `ohlc` table,
    restricted to that ticker's first..last date in `data` (its active history).
    """
    span = data.groupby("ticker")["ts"].agg(["min", "max"])
    ref = reference.merge(span, left_on="ticker", right_index=True)
    ref = ref.loc[ref["ts"].between(ref["min"], ref["max"]), ["ticker", "ts"]]
    merged = ref.merge(data[["ticker", "ts"]].drop_duplicates(), how="left", indicator=True)
    return merged.loc[merged["_merge"] == "left_only", ["ticker", "ts"]].reset_index(drop=True)

def gap_summary(gaps: pd.DataFrame, data: pd.DataFrame) -> list[str]:
    """One sentence per ticker: 'Ticker X is missing Y trading days between start and end'."""
    span = data.groupby("ticker")["ts"].agg(["min", "max"])
    counts = gaps.groupby("ticker").size().reindex(span.index, fill_value=0)
    return [f"Ticker {t} is missing {n} trading days between {span.loc[t, 'min']} and {span.loc[t, 'max']}"
            for t, n in counts.items()]

# Missing values per column
display(missing_cells(dedup))

# Trading calendar from the supplied reference table, with tickers/dates normalized the same way
with sqlite3.connect(DB_PATH) as conn:
    reference = pd.read_sql_query("SELECT DISTINCT ticker, ts FROM ohlc", conn)
reference["ticker"] = normalize_ticker(reference["ticker"])
reference["ts"] = pd.to_datetime(reference["ts"]).dt.strftime("%Y-%m-%d")

# Sessions the reference has but ohlc_hw does not
gaps = missing_sessions(dedup, reference)
print("Missing sessions in ohlc_hw:", len(gaps))
for line in gap_summary(gaps, dedup):
    print(line)

# Demonstrate the detector on one deliberately removed date
demo = dedup.loc[dedup["ticker"] == "AAPL"].sort_values("ts")
removed = demo["ts"].iloc[100]
demo_gaps = missing_sessions(demo.loc[demo["ts"] != removed], reference)
print("Injected gap:", removed, "→ detected:", demo_gaps["ts"].tolist())
assert demo_gaps["ts"].tolist() == [removed]

,missing,pct
ts,0,0.000000
open,0,0.000000
high,0,0.000000
low,0,0.000000
close,0,0.000000
adj_close,18374,0.049895
volume,0,0.000000
ticker,0,0.000000
raw_ticker,0,0.000000


Missing sessions in ohlc_hw: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 

## 5. Quality report: detect every issue
Each check returns a True/False mask per row. `quality_report` turns them into one row per (observation, issue) with a severity. This is the **before** picture; nothing is repaired yet.

Extreme return = a daily return more than 3 standard deviations from that ticker's mean. It is a flag for review, not proof of an error.

In [5]:
PRICE_COLS = ["open", "high", "low", "close"]

# high = value cannot be true; medium = may be true but needs review; low = plausible
SEVERITY = {
    "missing_session":    "high",
    "nonpositive_price":  "high",
    "nonfinite_price":    "high",
    "high_below_close":   "high",
    "low_above_close":    "high",
    "open_outside_range": "high",
    "high_below_low":     "high",
    "missing_adj_close":  "medium",
    "invalid_adj_close":  "high",
    "extreme_return":     "medium",
    "zero_volume":        "low",
}

def simple_returns(data: pd.DataFrame) -> pd.Series:
    """Daily simple return of raw close within each ticker; invalid closes give NaN."""
    valid = data["close"].where(np.isfinite(data["close"]) & data["close"].gt(0))
    return valid.groupby(data["ticker"]).pct_change(fill_method=None)

def issue_masks(data: pd.DataFrame) -> dict[str, pd.Series]:
    """Boolean mask per issue type. `data` must be sorted by ticker, ts."""
    p = data[PRICE_COLS]
    ret = simple_returns(data)
    g = ret.groupby(data["ticker"])
    mean, std = g.transform("mean"), g.transform("std")      # std uses ddof=1
    return {
        "nonfinite_price":    (~np.isfinite(p)).any(axis=1),
        "nonpositive_price":  p.le(0).any(axis=1),
        "high_below_close":   data["high"].lt(data["close"]),
        "low_above_close":    data["low"].gt(data["close"]),
        "open_outside_range": data["open"].lt(data["low"]) | data["open"].gt(data["high"]),
        "high_below_low":     data["high"].lt(data["low"]),
        "missing_adj_close":  data["adj_close"].isna(),
        "invalid_adj_close":  data["adj_close"].notna()
                              & (~np.isfinite(data["adj_close"]) | data["adj_close"].le(0)),
        "extreme_return":     ret.notna() & std.gt(0) & (ret - mean).abs().gt(3 * std),
        "zero_volume":        data["volume"].eq(0),
    }

def quality_report(data: pd.DataFrame, gaps: pd.DataFrame) -> pd.DataFrame:
    """One row per (observation, issue). An observation can have several issues."""
    data = data.sort_values(["ticker", "ts"]).reset_index(drop=True)
    ret = simple_returns(data)
    parts = []
    for issue, mask in issue_masks(data).items():
        rows = data.loc[mask, ["ticker", "ts"]].rename(columns={"ts": "issue_date"})
        if issue == "extreme_return":
            rows["description"] = ("Simple return " + ret[mask].map("{:+.1%}".format).astype(str)
                                   + " beyond 3σ of ticker history")
        else:
            rows["description"] = issue.replace("_", " ")
        rows["issue_type"] = issue
        parts.append(rows)
    if len(gaps):
        parts.append(gaps.rename(columns={"ts": "issue_date"})
                         .assign(issue_type="missing_session",
                                 description="date in ohlc reference but absent from ohlc_hw"))
    report = pd.concat(parts, ignore_index=True)
    report["severity"] = report["issue_type"].map(SEVERITY)
    return report[["ticker", "issue_type", "issue_date", "description", "severity"]]

# Run the checks on the de-duplicated data and summarize by issue type
dedup = dedup.sort_values(["ticker", "ts"]).reset_index(drop=True)
report_before = quality_report(dedup, gaps)     # the diagnosis BEFORE any repair; keep it
summary = (report_before.groupby(["issue_type", "severity"]).size()
                        .rename("rows").reset_index().sort_values("rows", ascending=False))
display(summary)
display(report_before.query("severity == 'high'").head(15))

,issue_type,severity,rows
1,missing_adj_close,medium,18374
0,extreme_return,medium,5056
4,zero_volume,low,75
2,nonpositive_price,high,5
3,open_outside_range,high,5


,ticker,issue_type,issue_date,description,severity
0,CVX,nonpositive_price,1962-01-02,nonpositive price,high
1,JNJ,nonpositive_price,1962-01-02,nonpositive price,high
2,MCD,nonpositive_price,1966-07-05,nonpositive price,high
3,MRK,nonpositive_price,1962-01-02,nonpositive price,high
4,PG,nonpositive_price,1962-01-02,nonpositive price,high
5,CVX,open_outside_range,1962-01-02,open outside range,high
6,JNJ,open_outside_range,1962-01-02,open outside range,high
7,MCD,open_outside_range,1966-07-05,open outside range,high
8,MRK,open_outside_range,1962-01-02,open outside range,high
9,PG,open_outside_range,1962-01-02,open outside range,high


## 6. Compare against the reference table
The supplied `ohlc` table is used as the trusted source.

- Raw open/high/low/close/volume match the reference on every row.
- The damage is in `adj_close`: missing values, and values whose **adjustment factor** (`adj_close / close`) differs from the reference's.
- All 5,056 flagged closes match the reference. Matching does not prove a move is real: §8 finds 3 one-day spikes present in both tables.

In [6]:
# Load the full reference table (same columns as ohlc_hw) and normalize keys the same way
with sqlite3.connect(DB_PATH) as conn:
    print(pd.read_sql_query("PRAGMA table_info(ohlc)", conn)[["name", "type"]].T)
    ohlc_ref = pd.read_sql_query("SELECT * FROM ohlc", conn)
ohlc_ref["ticker"] = normalize_ticker(ohlc_ref["ticker"])
ohlc_ref["ts"] = pd.to_datetime(ohlc_ref["ts"]).dt.strftime("%Y-%m-%d")

# Line every ohlc_hw row up with its reference row; *_ref columns come from ohlc
cmp = dedup.merge(ohlc_ref, on=["ticker", "ts"], how="left", suffixes=("", "_ref"))
print("Rows matched in reference:", cmp["close_ref"].notna().sum(), "of", len(cmp))
# Raw fields: count rows that differ from the reference
for col in ["open", "high", "low", "close", "volume"]:
    diff = ~np.isclose(cmp[col], cmp[f"{col}_ref"], rtol=1e-6) & cmp[f"{col}_ref"].notna()
    print(f"{col:>6}: {diff.sum():>6} rows differ from reference")

# Adjustment factor = adj_close / close. Compare the vendor's implied factor with the reference's.
cmp["factor"] = cmp["adj_close"] / cmp["close"]
cmp["factor_ref"] = cmp["adj_close_ref"] / cmp["close_ref"]
bad_factor = cmp["adj_close"].notna() & ~np.isclose(cmp["factor"], cmp["factor_ref"], rtol=1e-4)
print("adj_close factor differs from reference:", bad_factor.sum())
print("Missing adj_close with a usable reference factor:",
      (cmp["adj_close"].isna() & cmp["factor_ref"].gt(0)).sum(), "of", cmp["adj_close"].isna().sum())

# Do the flagged extreme returns / nonpositive prices also appear in the reference?
# If the reference has the same close, the move is in the source data, not a vendor error.
flag = report_before.loc[report_before["issue_type"].isin(["nonpositive_price", "extreme_return"]),
                         ["ticker", "issue_type", "issue_date"]]
chk = flag.merge(cmp, left_on=["ticker", "issue_date"], right_on=["ticker", "ts"])
chk["close_matches_ref"] = np.isclose(chk["close"], chk["close_ref"], rtol=1e-6)
# Show the evidence tables
display(chk.groupby("issue_type")["close_matches_ref"].value_counts().unstack(fill_value=0))
display(chk.loc[chk["issue_type"] == "nonpositive_price",
                ["ticker", "ts", "open", "open_ref", "high", "low", "close"]])
display(cmp.loc[bad_factor, ["ticker", "ts", "close", "adj_close", "factor", "factor_ref"]].head(10))

         0      1      2      3      4          5       6       7
name    ts   open   high    low  close  adj_close  volume  ticker
type  DATE  FLOAT  FLOAT  FLOAT  FLOAT      FLOAT  BIGINT    TEXT
Rows matched in reference: 368251 of 368251
  open:      0 rows differ from reference
  high:      0 rows differ from reference
   low:      0 rows differ from reference
 close:      0 rows differ from reference
volume:      0 rows differ from reference
adj_close factor differs from reference: 10417
Missing adj_close with a usable reference factor: 18374 of 18374


close_matches_ref,True
issue_type,
extreme_return,5056
nonpositive_price,5


,ticker,ts,open,open_ref,high,low,close
0,CVX,1962-01-02,0.0,0.0,3.30000,3.24000,3.30000
1,JNJ,1962-01-02,0.0,0.0,0.22338,0.22222,0.22338
2,MCD,1966-07-05,0.0,0.0,0.27366,0.26749,0.26955
3,MRK,1962-01-02,0.0,0.0,0.39703,0.39317,0.39537
4,PG,1962-01-02,0.0,0.0,1.44000,1.42000,1.43000


,ticker,ts,close,adj_close,factor,factor_ref
35,AAPL,1981-02-03,0.123330,1.264587,10.253683,0.770853
93,AAPL,1981-04-28,0.126120,2.036653,16.148537,0.770854
101,AAPL,1981-05-08,0.125000,2.129290,17.034321,0.770878
157,AAPL,1981-07-29,0.106030,0.506329,4.775336,0.770857
163,AAPL,1981-08-06,0.112720,2.009828,17.830273,0.770899
167,AAPL,1981-08-13,0.103800,1.122224,10.811404,0.770841
168,AAPL,1981-08-14,0.102120,1.171347,11.470295,0.770878
206,AAPL,1981-10-08,0.082589,1.245215,15.077244,0.770878
264,AAPL,1981-12-31,0.098772,1.730921,17.524412,0.770878
274,AAPL,1982-01-15,0.089286,1.577125,17.663741,0.770878


## 7. Inspect specific cases
Look at a few days around suspicious dates to see what really happened before deciding how to fix it.

In [7]:
def window(cmp: pd.DataFrame, ticker: str, date: str, days: int = 2) -> pd.DataFrame:
    """Rows for one ticker from `days` sessions before to `days` sessions after `date`."""
    t = cmp.loc[cmp["ticker"] == ticker].sort_values("ts").reset_index(drop=True)
    i = t.index[t["ts"] == date][0]
    return t.loc[max(i - days, 0): i + days,
                 ["ticker", "ts", "open", "high", "low", "close", "volume",
                  "adj_close", "adj_close_ref", "factor_ref"]]

# Hand-picked cases: two TRV spikes, an AAPL reference-factor blip, an MSFT special dividend
for ticker, date in [("TRV", "1976-10-28"), ("TRV", "1981-08-10"),
                     ("AAPL", "1990-10-19"), ("MSFT", "2004-11-15")]:
    display(window(cmp, ticker, date))

,ticker,ts,open,high,low,close,volume,adj_close,adj_close_ref,factor_ref
1150,TRV,1976-10-26,4.50,4.50,4.50,4.50,91200,0.95719,0.95719,0.212709
1151,TRV,1976-10-27,4.61,4.61,4.61,4.61,230400,13.88325,0.98046,0.212681
1152,TRV,1976-10-28,9.22,9.22,9.22,9.22,0,NaN,3.21000,0.348156
1153,TRV,1976-10-29,4.69,4.69,4.69,4.69,79200,0.99707,0.99707,0.212595
1154,TRV,1976-11-01,4.66,4.66,4.66,4.66,59200,0.99043,0.99043,0.212539


,ticker,ts,open,high,low,close,volume,adj_close,adj_close_ref,factor_ref
2356,TRV,1981-08-06,6.06,6.06,6.06,6.06,60000,1.58,1.58,0.260726
2357,TRV,1981-08-07,6.03,6.03,6.03,6.03,264800,1.57,1.57,0.260365
2358,TRV,1981-08-10,12.06,12.06,12.06,12.06,0,4.20,4.20,0.348259
2359,TRV,1981-08-11,6.03,6.03,6.03,6.03,24800,1.57,1.57,0.260365
2360,TRV,1981-08-12,6.13,6.13,6.13,6.13,267200,1.59,1.59,0.259380


,ticker,ts,open,high,low,close,volume,adj_close,adj_close_ref,factor_ref
2484,AAPL,1990-10-17,0.22321,0.23661,0.22321,0.23661,309064307,3.843573,0.18837,0.796120
2485,AAPL,1990-10-18,0.23661,0.25670,0.23661,0.25446,314888312,0.202590,0.20259,0.796157
2486,AAPL,1990-10-19,0.28000,0.28000,0.27000,0.28000,933733760,0.280000,0.28000,1.000000
2487,AAPL,1990-10-22,0.28013,0.28125,0.27232,0.27790,252739452,0.221250,0.22125,0.796150
2488,AAPL,1990-10-23,0.27790,0.28125,0.27009,0.27679,167048164,0.220360,0.22036,0.796127


,ticker,ts,open,high,low,close,volume,adj_close,adj_close_ref,factor_ref
4711,MSFT,2004-11-11,29.87,30.08,29.82,29.98,88754000,18.70,18.70,0.623749
4712,MSFT,2004-11-12,30.17,30.20,29.80,29.97,162601376,18.70,18.70,0.623957
4713,MSFT,2004-11-15,27.36,27.50,27.20,27.39,104734064,19.05,19.05,0.695509
4714,MSFT,2004-11-16,27.33,27.34,27.05,27.12,64592720,18.86,18.86,0.695428
4715,MSFT,2004-11-17,27.25,27.36,27.06,27.17,59091056,18.89,18.89,0.695252


## 8. Evidence flags: price spikes and factor changes
- **spike:** a one-day jump that fully reverses the next day. This is a data error.
- **factor_blip:** the reference factor jumps for one day, then returns. This is an error in the reference.
- **factor_event:** the factor steps to a new level and stays there. This is a usually a split or special dividend.

In [8]:
def spike_and_factor_flags(cmp: pd.DataFrame, jump: float = 0.30, revert: float = 0.05,
                           factor_tol: float = 0.05, same_tol: float = 0.01) -> pd.DataFrame:
    """Add evidence columns for one-day price spikes and one-day reference-factor blips.

    spike:         close moves > `jump` vs the previous day, and the next close is back
                   within `revert` of the previous close (the move fully reverses).
    factor_blip:   factor_ref differs by > `factor_tol` from both neighbours, while the
                   neighbours agree within `same_tol` (no lasting corporate action).
    factor_event:  factor_ref changes by > `factor_tol` and stays at the new level next day.
    """
    d = cmp.sort_values(["ticker", "ts"]).reset_index(drop=True)
    g = d.groupby("ticker")
    c_prev, c_next = g["close"].shift(1), g["close"].shift(-1)
    f_prev, f_next = g["factor_ref"].shift(1), g["factor_ref"].shift(-1)
    d["spike"] = ((d["close"] / c_prev - 1).abs().gt(jump)
                  & (c_next / c_prev - 1).abs().lt(revert))
    neighbours_agree = (f_next / f_prev - 1).abs().lt(same_tol)
    d["factor_blip"] = ((d["factor_ref"] / f_prev - 1).abs().gt(factor_tol)
                        & (d["factor_ref"] / f_next - 1).abs().gt(factor_tol)
                        & neighbours_agree)
    after_blip = d.groupby("ticker")["factor_blip"].shift(1, fill_value=False).astype(bool)
    d["factor_event"] = ((d["factor_ref"] / f_prev - 1).abs().gt(factor_tol)
                         & (f_next / d["factor_ref"] - 1).abs().lt(same_tol)
                         & ~after_blip)
    return d

# Add the evidence columns and show how many of each were found
cmp = spike_and_factor_flags(cmp)
print("Price spikes:", cmp["spike"].sum(),
      "| of which zero volume:", (cmp["spike"] & cmp["volume"].eq(0)).sum())
print("Reference factor blips:", cmp["factor_blip"].sum())
print("Lasting factor events (splits/special dividends):", cmp["factor_event"].sum())
display(cmp.loc[cmp["spike"] | cmp["factor_blip"] | cmp["factor_event"],
                ["ticker", "ts", "close", "volume", "factor_ref",
                 "spike", "factor_blip", "factor_event"]].head(20))

Price spikes: 3 | of which zero volume: 3
Reference factor blips: 52
Lasting factor events (splits/special dividends): 31


,ticker,ts,close,volume,factor_ref,spike,factor_blip,factor_event
2486,AAPL,1990-10-19,0.28000,933733760,1.000000,False,True,False
58955,CAT,1968-01-02,3.45000,93592,0.301449,False,False,True
104457,DIS,1968-01-02,0.28334,711679,0.620774,False,False,True
105948,DIS,1974-01-08,0.77327,6724379,0.663352,False,True,False
105949,DIS,1974-01-09,0.75385,4962521,0.620773,False,True,False
105952,DIS,1974-01-13,0.80324,2096568,0.620773,False,False,True
105955,DIS,1974-01-16,0.84007,2418246,0.663349,False,False,True
105958,DIS,1974-01-20,0.91502,2962125,0.620762,False,False,True
105960,DIS,1974-01-22,0.88888,3658996,0.663352,False,True,False
105961,DIS,1974-01-23,0.93062,2611094,0.620769,False,True,False


## 9. Clean the data
Repairs are applied in order. Each one records what it did in `price_repair` / `adj_close_repair`.
Rows that cannot be repaired honestly go to quarantine with a reason. Values are never borrowed from another ticker.

`adj_close` is rebuilt as `factor × close` using the reference factor for that date.

In [9]:
def interpolate_interior(s: pd.Series) -> pd.Series:
    """Linear interpolation between valid neighbours only (no leading/trailing extrapolation)."""
    return s.interpolate(method="linear", limit_area="inside")

def clean_ohlc(cmp: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Repair dedup'd rows (with reference columns from Steps 8 and 11). Return (clean, quarantine)."""
    d = cmp.sort_values(["ticker", "ts"]).reset_index(drop=True).copy()
    d["price_repair"] = ""
    d["adj_close_repair"] = ""
    quarantine = []

    # 1. Drop rows where ALL raw prices are missing (before any fill)
    all_missing = d[PRICE_COLS].isna().all(axis=1)
    quarantine.append(d.loc[all_missing].assign(reason="all raw prices missing"))
    d = d.loc[~all_missing].reset_index(drop=True)

    # 2. Invalid raw values -> NaN (nonpositive / nonfinite prices)
    bad = ~np.isfinite(d[PRICE_COLS]) | d[PRICE_COLS].le(0)
    d[PRICE_COLS] = d[PRICE_COLS].mask(bad)
    d.loc[bad.any(axis=1), "price_repair"] += "invalid_value;"

    # 3. Confirmed one-day spikes (whole row, reverts next day) -> interpolate from neighbours
    spike = d["spike"]
    for col in PRICE_COLS:
        interp = d[col].mask(spike).groupby(d["ticker"]).transform(interpolate_interior)
        d.loc[spike, col] = interp[spike]          # only confirmed errors are interpolated
    d.loc[spike, "price_repair"] += "spike_interpolated;"

    # 4. Impossible OHLC relationships -> Option 1: set the offending field to NaN, forward-fill
    hi_bad = d["high"].lt(d[["open", "close"]].max(axis=1)) | d["high"].lt(d["low"])
    lo_bad = d["low"].gt(d[["open", "close"]].min(axis=1))
    d.loc[hi_bad, "high"] = np.nan
    d.loc[lo_bad, "low"] = np.nan
    d.loc[hi_bad | lo_bad, "price_repair"] += "ohlc_bound_ffill;"

    # 5. Remaining missing prices -> forward-fill within ticker (carry last known price)
    was_na = d[PRICE_COLS].isna().any(axis=1)
    d[PRICE_COLS] = d.groupby("ticker")[PRICE_COLS].ffill()
    d.loc[was_na & d[PRICE_COLS].notna().all(axis=1), "price_repair"] += "ffill;"

    # 6. Re-check bounds after filling: widen high/low to contain open/close (never move close)
    hi = d[["open", "high", "close"]].max(axis=1)
    lo = d[["open", "low", "close"]].min(axis=1)
    widened = d["high"].ne(hi) | d["low"].ne(lo)
    d["high"], d["low"] = hi, lo
    d.loc[widened & d[PRICE_COLS].notna().all(axis=1), "price_repair"] += "bounds_widened;"

    # 7. Unresolved (leading gaps) -> quarantine with reason, never borrow another ticker's value
    unresolved = d[PRICE_COLS].isna().any(axis=1)
    quarantine.append(d.loc[unresolved].assign(reason="no earlier valid price (leading gap)"))
    d = d.loc[~unresolved].reset_index(drop=True)

    # 8. Volume: unknown -> 0 with flag (unknown is not evidence of no trading)
    d["volume_repaired"] = d["volume"].isna()
    d["volume"] = d["volume"].fillna(0).astype("int64")

    # 9. adj_close: trusted date-specific factor from reference; one-day blips use neighbour factor
    factor = d["factor_ref"].mask(d["factor_blip"])
    factor = factor.groupby(d["ticker"]).transform(lambda s: s.ffill())   # neighbours agree by definition
    vendor_ok = (d["adj_close"].notna() & np.isfinite(d["adj_close"]) & d["adj_close"].gt(0)
                 & np.isclose(d["adj_close"] / d["close"], factor, rtol=1e-4)
                 & d["price_repair"].eq(""))
    price_fixed = d["price_repair"].ne("")
    d.loc[~vendor_ok & d["adj_close"].isna(), "adj_close_repair"] = "missing->factor*close"
    d.loc[~vendor_ok & d["adj_close"].notna() & ~price_fixed, "adj_close_repair"] = "inconsistent->factor*close"
    d.loc[~vendor_ok & d["adj_close"].notna() & price_fixed, "adj_close_repair"] = "raw_repaired->factor*close"
    d.loc[d["factor_blip"] & ~vendor_ok, "adj_close_repair"] += ";ref_factor_blip->neighbour"
    d["adj_close"] = d["adj_close"].where(vendor_ok, factor * d["close"])
    unresolved_adj = d["adj_close"].isna() | ~np.isfinite(d["adj_close"]) | d["adj_close"].le(0)
    quarantine.append(d.loc[unresolved_adj].assign(reason="no trusted adjustment factor"))
    d = d.loc[~unresolved_adj].reset_index(drop=True)

    d["price_repaired"] = d["price_repair"].ne("")
    d["adj_close_repaired"] = d["adj_close_repair"].ne("")
    q = pd.concat(quarantine, ignore_index=True)
    return d, q

# Apply the repairs, then show what was quarantined and how many rows each repair touched
clean, quarantined = clean_ohlc(cmp)
print("Input rows:", len(cmp), "| clean:", len(clean), "| quarantined:", len(quarantined))
display(quarantined[["ticker", "ts", "open", "close", "reason"]])
display(clean["price_repair"].value_counts())
display(clean["adj_close_repair"].value_counts())
print("Volume repaired:", clean["volume_repaired"].sum())

Input rows: 368251 | clean: 368246 | quarantined: 5


,ticker,ts,open,close,reason
0,CVX,1962-01-02,NaN,3.30000,no earlier valid price (leading gap)
1,JNJ,1962-01-02,NaN,0.22338,no earlier valid price (leading gap)
2,MCD,1966-07-05,NaN,0.26955,no earlier valid price (leading gap)
3,MRK,1962-01-02,NaN,0.39537,no earlier valid price (leading gap)
4,PG,1962-01-02,NaN,1.43000,no earlier valid price (leading gap)


price_repair
                       368243
spike_interpolated;         3
Name: count, dtype: int64

adj_close_repair
                                                         339416
missing->factor*close                                     18371
inconsistent->factor*close                                10416
inconsistent->factor*close;ref_factor_blip->neighbour        38
missing->factor*close;ref_factor_blip->neighbour              3
raw_repaired->factor*close;ref_factor_blip->neighbour         2
Name: count, dtype: int64

Volume repaired: 0


In [10]:
display(quarantined["reason"].value_counts()
        .reindex(["all raw prices missing", "no earlier valid price (leading gap)",
                  "no trusted adjustment factor"], fill_value=0).rename("quarantined_rows"))

reason
all raw prices missing                  0
no earlier valid price (leading gap)    5
no trusted adjustment factor            0
Name: quarantined_rows, dtype: int64

## 10. Validate the cleaned data
Assert every rule (no missing or nonpositive prices, no duplicate keys, open/close inside [low, high], ISO dates). Then compare issue counts before vs after, and check that every raw row is accounted for.

In [11]:
def validate(clean: pd.DataFrame) -> None:
    """Assert every Part 1.3 validation rule; raise with a clear message on failure."""
    prices = clean[PRICE_COLS + ["adj_close"]]
    assert prices.notna().all().all(), "missing price"
    assert np.isfinite(prices).all().all(), "non-finite price"
    assert prices.gt(0).all().all(), "nonpositive price"
    assert not clean.duplicated(["ticker", "ts"]).any(), "duplicate (ticker, ts)"
    assert np.isfinite(clean["volume"]).all() and clean["volume"].ge(0).all(), "bad volume"
    assert clean["open"].between(clean["low"], clean["high"]).all(), "open outside [low, high]"
    assert clean["close"].between(clean["low"], clean["high"]).all(), "close outside [low, high]"
    assert pd.to_datetime(clean["ts"], format="%Y-%m-%d").notna().all(), "ts not ISO"
    print("All validation checks passed.")

validate(clean)          # stops the notebook if any rule fails

# Review flags kept with the data (not deletions)
masks_after = issue_masks(clean)
clean["extreme_return_flag"] = masks_after["extreme_return"].to_numpy()
clean["zero_volume_flag"] = masks_after["zero_volume"].to_numpy()

# Re-run the same quality checks on the cleaned data: before vs after counts per issue
report_after = quality_report(clean, missing_sessions(clean, reference))
before_after = (report_before.groupby("issue_type").size().rename("before").to_frame()
                .join(report_after.groupby("issue_type").size().rename("after"), how="outer")
                .fillna(0).astype(int))
display(before_after)

# Summary statistics before vs after cleaning
cols = PRICE_COLS + ["adj_close", "volume"]
stats = pd.concat({"before (dedup)": dedup[cols].describe().T[["count", "min", "max", "mean"]],
                   "after (clean)":  clean[cols].describe().T[["count", "min", "max", "mean"]]},
                  axis=1)
display(stats)

# Every raw row must be accounted for: removed as duplicate, quarantined, or retained
accounting = pd.Series({
    "raw rows (ohlc_hw)": len(raw),
    "exact duplicates removed": int(exact_dup.sum()),
    "quarantined": len(quarantined),
    "retained (ohlc_hw_cleaned)": len(clean),
})
display(accounting.to_frame("rows"))
assert accounting.iloc[0] == accounting.iloc[1:].sum(), "row accounting does not balance"

All validation checks passed.


,before,after
issue_type,,
extreme_return,5056,5176
missing_adj_close,18374,0
nonpositive_price,5,0
open_outside_range,5,0
zero_volume,75,74


before (dedup)                                       after (clean)  \
                   count       min           max          mean         count   
open            368251.0  0.000000  6.042600e+02  4.630877e+01      368246.0   
high            368251.0  0.005208  6.086300e+02  4.677120e+01      368246.0   
low             368251.0  0.004801  5.978800e+02  4.583977e+01      368246.0   
close           368251.0  0.005208  6.054000e+02  4.631671e+01      368246.0   
adj_close       349877.0  0.002945  1.044251e+04  5.171154e+01      368246.0   
volume          368251.0  0.000000  9.276606e+09  3.095142e+07      368246.0   

                                                 
                min           max          mean  
open       0.005208  6.042600e+02  4.630935e+01  
high       0.005208  6.086300e+02  4.677178e+01  
low        0.004801  5.978800e+02  4.584034e+01  
close      0.005208  6.054000e+02  4.631729e+01  
adj_close  0.002945  6.054000e+02  3.727465e+01  
volume     0.000000  9.276606e+09  3.095184e+07

,rows
raw rows (ohlc_hw),397711
exact duplicates removed,29460
quarantined,5
retained (ohlc_hw_cleaned),368246


## 11. Adjusted-price check and evidence notes
**Implied factor** = `adj_close / close`. After all repairs it must equal the reference factor on every row; the only exceptions are the reference's one-day blips, where the neighbouring day's factor is used.

Each extreme return that is kept gets a short note (`return_note`) explaining why it was kept.

In [12]:
# Adjusted-price consistency after all repairs: adj_close / close must equal the trusted factor
implied = clean["adj_close"] / clean["close"]
consistent = np.isclose(implied, clean["factor_ref"], rtol=1e-4) | clean["factor_blip"]
assert consistent.all(), f"{(~consistent).sum()} rows with inconsistent adjustment factor"
print(f"Adjusted-price consistency passed; implied factor range {implied.min():.4f} to {implied.max():.4f}")

# Evidence note for every kept extreme-return flag (stored with the cleaned data)
clean["return_note"] = ""
clean.loc[clean["extreme_return_flag"] & clean["factor_event"], "return_note"] = (
    "reference adjustment-factor step on this date (split/special dividend); raw move, adjusted continuous")
clean.loc[clean["extreme_return_flag"] & ~clean["factor_event"], "return_note"] = (
    "kept: close matches reference; cause not verified (no news source)")
clean.loc[clean["price_repair"].str.contains("spike"), "return_note"] = (
    "confirmed one-day spike (x2 whole row, zero volume, reverts) -> interpolated")
display(clean["return_note"].value_counts())

Adjusted-price consistency passed; implied factor range 0.1018 to 1.0000


return_note
                                                                                                         363067
kept: close matches reference; cause not verified (no news source)                                         5174
confirmed one-day spike (x2 whole row, zero volume, reverts) -> interpolated                                  3
reference adjustment-factor step on this date (split/special dividend); raw move, adjusted continuous         2
Name: count, dtype: int64

## 12. Save to the database
Write `ohlc_hw_cleaned`, `ohlc_hw_quarantine` and `ohlc_hw_quality_report`. Then read them back to confirm the save worked and that raw `ohlc_hw` is unchanged.

In [13]:
# Columns written to ohlc_hw_cleaned: data, original label, repair audit trail, review flags
SAVE_COLS = (["ts", "ticker"] + PRICE_COLS + ["adj_close", "volume", "raw_ticker",
              "price_repaired", "price_repair", "adj_close_repaired", "adj_close_repair",
              "volume_repaired", "extreme_return_flag", "zero_volume_flag",
              "return_note"])
to_save = clean[SAVE_COLS].astype({"raw_ticker": str, "ticker": str})

# Write the three output tables (replace = re-running the notebook is safe)
with sqlite3.connect(DB_PATH) as conn:
    to_save.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
    (quarantined[["ts", "ticker"] + PRICE_COLS + ["adj_close", "volume", "raw_ticker", "reason"]]
        .astype({"raw_ticker": str, "ticker": str})
        .to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False))
    report_before.astype({"ticker": str}).to_sql("ohlc_hw_quality_report", conn,
                                                 if_exists="replace", index=False)

    # Read back from the database to prove the save worked and the raw table is unchanged
    back = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", conn)
    raw_count = pd.read_sql_query("SELECT COUNT(*) AS n FROM ohlc_hw", conn)["n"].iloc[0]
    # Parameterized query (? placeholders) for a small sample
    sample = pd.read_sql_query(
        "SELECT ts, close, adj_close FROM ohlc_hw_cleaned "
        "WHERE ticker = ? AND ts BETWEEN ? AND ? ORDER BY ts",
        conn, params=("AAPL", "2024-06-01", "2024-06-07"))

assert len(back) == len(clean)
assert raw_count == len(raw), "raw table changed!"
validate(back)
print("Saved", len(back), "rows; raw ohlc_hw still has", raw_count, "rows.")
display(sample)

All validation checks passed.
Saved 368246 rows; raw ohlc_hw still has 397711 rows.


,ts,close,adj_close
0,2024-06-03,194.03,193.81
1,2024-06-04,194.35,194.13
2,2024-06-05,195.87,195.64
3,2024-06-06,194.48,194.26
4,2024-06-07,196.89,196.66


## 13. Test on an injected example
The real data has no high < close, high < low, infinite prices or missing sessions. A small fake ticker with each problem planted on purpose shows that the detectors find them and that cleaning fixes or quarantines them.

In [14]:
# Small injected example for issue types that are absent from the real data
dates = list(pd.bdate_range("2024-03-04", periods=8).strftime("%Y-%m-%d"))
toy = pd.DataFrame({"ticker": "TOY", "ts": dates,
                    "open": 10.0, "high": 11.0, "low": 9.5,
                    "close": [10.0, 10.1, 10.2, 10.3, 10.4, 10.5, 10.6, 10.7],
                    "volume": 1000.0, "factor_ref": 0.5})
toy.loc[1, "high"] = 9.9                       # high < close
toy.loc[2, "low"] = 10.4                       # low > close
toy.loc[3, ["high", "low"]] = [9.0, 11.0]      # high < low
toy.loc[4, "close"] = np.inf                   # non-finite price
toy.loc[6, PRICE_COLS] = np.nan                # all raw prices missing
toy.loc[7, "volume"] = np.nan                  # unknown volume
toy["adj_close"] = toy["close"] * toy["factor_ref"]
toy_ref = toy[["ticker", "ts"]].copy()         # reference calendar has all 8 dates
toy = toy.loc[toy["ts"] != dates[5]].reset_index(drop=True)   # missing session

# The detectors must find every injected issue...
toy_gaps = missing_sessions(toy, toy_ref)
toy_report = quality_report(toy, toy_gaps)
display(toy_report[["issue_type", "issue_date", "severity"]])

# ...and the cleaning must repair or quarantine them so validation passes
toy_clean, toy_q = clean_ohlc(spike_and_factor_flags(toy.assign(adj_close_ref=toy["adj_close"])))
display(toy_clean[["ts"] + PRICE_COLS + ["adj_close", "volume", "price_repair", "volume_repaired"]])
display(toy_q[["ts", "reason"]])
validate(toy_clean)

expected = {"high_below_close", "low_above_close", "high_below_low",
            "nonfinite_price", "missing_session"}
assert expected <= set(toy_report["issue_type"]), expected - set(toy_report["issue_type"])
assert toy_clean["volume_repaired"].any() and len(toy_q) == 1
print("Injected-example checks passed.")

,issue_type,issue_date,severity
0,nonfinite_price,2024-03-08,high
1,nonfinite_price,2024-03-12,high
2,high_below_close,2024-03-05,high
3,high_below_close,2024-03-07,high
4,high_below_close,2024-03-08,high
5,low_above_close,2024-03-06,high
6,low_above_close,2024-03-07,high
7,open_outside_range,2024-03-05,high
8,open_outside_range,2024-03-06,high
9,open_outside_range,2024-03-07,high


,ts,open,high,low,close,adj_close,volume,price_repair,volume_repaired
0,2024-03-04,10.0,11.0,9.5,10.0,5.00,1000,,False
1,2024-03-05,10.0,11.0,9.5,10.1,5.05,1000,ohlc_bound_ffill;ffill;,False
2,2024-03-06,10.0,11.0,9.5,10.2,5.10,1000,ohlc_bound_ffill;ffill;,False
3,2024-03-07,10.0,11.0,9.5,10.3,5.15,1000,ohlc_bound_ffill;ffill;,False
4,2024-03-08,10.0,11.0,9.5,10.3,5.15,1000,invalid_value;ffill;,False
5,2024-03-13,10.0,11.0,9.5,10.7,5.35,0,,True


,ts,reason
0,2024-03-12,all raw prices missing


All validation checks passed.
Injected-example checks passed.
